# Book 0 — Extract OSM Weak-Label Sources and Road Morphology

This notebook downloads and parses OpenStreetMap data for the same urban-core AOIs used by Book 1. It runs in Google Colab because it uses `osmium-tool` and `pyrosm` to process the China PBF file efficiently.

**Input**: `urban_cores_by_builtup.geojson` from the pre-process notebook and the Geofabrik China OSM PBF.  
**Output**: per-city OSM candidate GeoJSONs, road GeoJSONs, and summary CSVs.  
**Runtime**: approximately 20-40 minutes on Colab after the China PBF is cached; the first run may take longer because the PBF is about 1.5 GB.  
**Role in the pipeline**: provides the weak supervision signal (`place=village/hamlet/locality` and `residential=rural`) and road-density morphology features for Book 2.


### Step 1: Mount Google Drive

**Purpose**: Connect Colab to the shared project directory where the AOI, PBF cache, and exported OSM files are stored.  
**Input**: Google Drive authentication.  
**Output**: `/content/drive` mounted for all subsequent file operations.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 2: Install and verify OSM processing tools

**Purpose**: Install the libraries used to slice and parse OpenStreetMap data.  
**Input**: Colab runtime.  
**Output**: `pyrosm`, `geopandas`, `shapely`, and native `osmium-tool` available for city-level extraction.


In [ ]:
# ----------------------------------------------------------------------------
# 1. Install and import
# ----------------------------------------------------------------------------
# pyrosm reads OSM PBF in pure Python. osmium-tool is a native C++ slicer used
# to pre-extract per-city PBFs, which is ~100x faster than letting pyrosm scan
# the full 1.5 GB file for every query.

!pip -q install pyrosm geopandas shapely
!apt-get -qq install osmium-tool > /dev/null 2>&1

import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import pyrosm
from shapely.geometry import box
from shapely.ops import unary_union

warnings.filterwarnings("ignore")

try:
    from importlib.metadata import version as _pkg_version
    print("pyrosm version:", _pkg_version("pyrosm"))
except Exception:
    print("pyrosm imported (version attribute not exposed in this build)")

print("osmium available:", os.system("osmium --version > /dev/null 2>&1") == 0)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.1 MB/s eta 0:00:00
pyrosm version: 0.6.2
osmium available: True


### Step 3: Define project folders and required AOI path

**Purpose**: Establish the Drive folder structure for the PBF cache, per-city outputs, and run logs.  
**Input**: project root and the urban-core AOI GeoJSON produced in pre-processing.  
**Output**: path variables used consistently by the OSM extraction workflow.


In [ ]:
# ----------------------------------------------------------------------------
# 2. Mount Drive and define paths
# ----------------------------------------------------------------------------
# Expected layout (matches Book 1 + Book 2):
#
# /content/drive/MyDrive/0069/week10/
#   data/
#     processed/city_districts/
#       urban_cores_by_builtup.geojson   <- shared AOI with Book 1
#     osm_pbf/
#       china-latest.osm.pbf
#       guangzhou.osm.pbf  ...
#   osm_candidates/
#     guangzhou_candidates.geojson  ...
#   osm_roads/
#     guangzhou_roads.geojson  ...
#   osm_logs/
#     book0_candidate_summary.csv
#     book0_road_summary.csv

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/0069/week10")
DATA_DIR = PROJECT_DIR / "data"
PBF_DIR = DATA_DIR / "osm_pbf"
OSM_CANDIDATE_DIR = PROJECT_DIR / "osm_candidates"
OSM_ROAD_DIR = PROJECT_DIR / "osm_roads"
LOG_DIR = PROJECT_DIR / "osm_logs"

for p in [DATA_DIR, PBF_DIR, OSM_CANDIDATE_DIR, OSM_ROAD_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PBF_PATH = PBF_DIR / "china-latest.osm.pbf"

# This is the new shared AOI file (output of
# scripts/prepare_urban_aoi_from_gadm_l3.ipynb). One row per L3 district,
# with a 'city' column. Multiple rows per city are unioned below.
URBAN_CORES_PATH = DATA_DIR / "processed_urbancontext" / "urban_cores_by_builtup.geojson"

# Legacy GADM file kept only for QA. No longer used as primary AOI.
LEGACY_GADM_PATH = DATA_DIR / "china_interest_city_boundaries_gadm41.geojson"

print("Project dir:", PROJECT_DIR)
print("Master PBF target:", PBF_PATH)
print("Urban cores AOI:", URBAN_CORES_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project dir: /content/drive/MyDrive/0069/week10
Master PBF target: /content/drive/MyDrive/0069/week10/data/osm_pbf/china-latest.osm.pbf
Urban cores AOI: /content/drive/MyDrive/0069/week10/data/processed_urbancontext/urban_cores_by_builtup.geojson


### Step 4: Download or reuse the China OSM PBF

**Purpose**: Cache the national OSM extract locally on Drive so city-level parsing is reproducible and does not depend on Overpass API rate limits.  
**Input**: Geofabrik China PBF URL.  
**Output**: `china-latest.osm.pbf` stored under the project data directory.


In [ ]:
# ----------------------------------------------------------------------------
# 3. Download China PBF (one-time, ~1.5 GB)
# ----------------------------------------------------------------------------

GEOFABRIK_URL = "https://download.geofabrik.de/asia/china-latest.osm.pbf"

if PBF_PATH.exists() and PBF_PATH.stat().st_size > 1_000_000_000:
    size_gb = PBF_PATH.stat().st_size / 1e9
    print(f"PBF already cached: {PBF_PATH}")
    print(f"  size: {size_gb:.2f} GB. Skipping download.")
else:
    print(f"Downloading {GEOFABRIK_URL}")
    print("This is a one-time ~1.5 GB download. Expect 5-15 minutes.")
    rc = os.system(f"wget -c -O {PBF_PATH} {GEOFABRIK_URL}")
    if rc != 0:
        raise RuntimeError(f"wget failed with exit code {rc}")
    print(f"Downloaded. size: {PBF_PATH.stat().st_size / 1e9:.2f} GB")


PBF already cached: /content/drive/MyDrive/0069/week10/data/osm_pbf/china-latest.osm.pbf
  size: 1.52 GB. Skipping download.


### Step 5: Load the urban-core AOI and build city polygons

**Purpose**: Union the retained Level-3 districts into one polygon or multipolygon per study city.  
**Input**: `urban_cores_by_builtup.geojson`.  
**Output**: `CITY_POLYGONS`, a dictionary of city-level AOI geometries used for PBF slicing and final clipping.


In [ ]:
# ----------------------------------------------------------------------------
# 4. Load shared urban-core AOI and build per-city polygons
# ----------------------------------------------------------------------------
# urban_cores_by_builtup.geojson holds one polygon per kept L3 district. A
# city can have many rows; we unary_union them per city to get a single
# (Multi)Polygon that matches Book 1's grid AOI exactly.

CITIES_TO_RUN = [
    "guangzhou", "shenzhen", "tianjin", "beijing", "xian",
    "dongguan", "shanghai", "chengdu", "chongqing", "wuhan",
]

OVERWRITE_PBF_EXTRACTS = True   # turn on to re-extract since AOI changed
OVERWRITE_GEOJSON = True

if not URBAN_CORES_PATH.exists():
    raise FileNotFoundError(
        f"Urban cores AOI not found: {URBAN_CORES_PATH}\n"
        f"  Run scripts/prepare_urban_aoi_from_gadm_l3.ipynb to generate it, "
        f"then upload urban_cores_by_builtup.geojson to:\n"
        f"  /content/drive/MyDrive/0069/week10/data/processed/city_districts/"
    )

urban_cores = gpd.read_file(URBAN_CORES_PATH).to_crs("EPSG:4326")
missing = sorted(set(CITIES_TO_RUN) - set(urban_cores["city"]))
if missing:
    raise ValueError(f"urban_cores is missing cities: {missing}")

print(f"Loaded urban_cores with {len(urban_cores)} L3-district rows "
      f"across {urban_cores['city'].nunique()} cities\n")

CITY_POLYGONS = {}
for city in CITIES_TO_RUN:
    sub = urban_cores[urban_cores["city"] == city]
    poly = unary_union(sub.geometry.tolist())
    if poly.is_empty:
        raise ValueError(f"{city}: empty polygon after union of L3 districts")
    CITY_POLYGONS[city] = poly
    area_km2 = gpd.GeoSeries([poly], crs="EPSG:4326").to_crs("EPSG:3857").area.iloc[0] / 1e6
    bounds = [round(b, 2) for b in poly.bounds]
    n_parts = len(sub)
    print(f"  {city:>10s}: {n_parts:>2d} district(s) -> "
          f"area={area_km2:>5.0f} km²  bounds={bounds}")


Loaded urban_cores with 84 L3-district rows across 10 cities

   guangzhou:  9 district(s) -> area= 4298 km²  bounds=[112.95, 22.56, 113.7, 23.62]
    shenzhen:  9 district(s) -> area= 2298 km²  bounds=[113.75, 22.45, 114.62, 22.86]
     tianjin: 11 district(s) -> area= 7378 km²  bounds=[116.88, 38.55, 118.06, 39.36]
     beijing:  9 district(s) -> area= 7355 km²  bounds=[116.04, 39.44, 116.98, 40.31]
        xian:  8 district(s) -> area= 2005 km²  bounds=[108.79, 34.17, 109.43, 34.74]
    dongguan:  1 district(s) -> area= 2902 km²  bounds=[113.51, 22.66, 114.26, 23.14]
    shanghai: 15 district(s) -> area= 7808 km²  bounds=[120.85, 30.69, 122.04, 31.55]
     chengdu:  9 district(s) -> area= 3719 km²  bounds=[103.68, 30.23, 104.33, 30.97]
   chongqing:  6 district(s) -> area= 1907 km²  bounds=[106.25, 29.26, 106.89, 29.75]
       wuhan:  7 district(s) -> area= 1309 km²  bounds=[114.12, 30.38, 114.64, 30.7]


### Step 6: Pre-extract per-city PBF files with osmium

**Purpose**: Create smaller city-specific PBF files using each city AOI bounding box, making later `pyrosm` parsing much faster.  
**Input**: national China PBF and city polygon bounds.  
**Output**: one cached PBF per city in the project PBF directory.


In [ ]:
# ----------------------------------------------------------------------------
# 5. Pre-extract per-city PBFs with osmium (using polygon bbox)
# ----------------------------------------------------------------------------
# osmium needs a rectangular bbox. We use the tight bounding box of each
# city's polygon (which may be larger than the old CORE_BBOXES because L3
# districts extend further in some cities). The polygon-shaped clip happens
# in step 7 via pyrosm output.

def osmium_extract_city(city, bbox, src_pbf, out_dir, overwrite=False):
    out_path = out_dir / f"{city}.osm.pbf"
    if out_path.exists() and out_path.stat().st_size > 100_000 and not overwrite:
        return out_path, True
    bbox_str = f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}"
    cmd = f"osmium extract -b {bbox_str} {src_pbf} -o {out_path} --overwrite"
    rc = os.system(cmd)
    if rc != 0 or not out_path.exists():
        return None, False
    return out_path, False


print("Pre-extracting per-city PBFs (bbox from polygon)...\n")
city_pbf_paths = {}
for city in CITIES_TO_RUN:
    t0 = time.time()
    extract_bbox = list(CITY_POLYGONS[city].bounds)
    pbf_path, cached = osmium_extract_city(
        city, extract_bbox, PBF_PATH, PBF_DIR,
        overwrite=OVERWRITE_PBF_EXTRACTS,
    )
    if pbf_path is None:
        print(f"  {city}: FAILED to extract")
        continue
    size_mb = pbf_path.stat().st_size / 1e6
    tag = "(cached)" if cached else f"({time.time()-t0:.1f}s)"
    print(f"  {city}: {size_mb:.1f} MB  {tag}")
    city_pbf_paths[city] = pbf_path


Pre-extracting per-city PBFs (bbox from polygon)...

  guangzhou: 26.2 MB  (114.3s)
  shenzhen: 26.2 MB  (102.9s)
  tianjin: 11.3 MB  (103.1s)
  beijing: 24.8 MB  (105.8s)
  xian: 16.6 MB  (100.6s)
  dongguan: 11.5 MB  (98.7s)
  shanghai: 28.4 MB  (102.7s)
  chengdu: 12.4 MB  (98.4s)
  chongqing: 7.9 MB  (103.4s)
  wuhan: 6.7 MB  (99.5s)


### Step 7: Define OSM parsing functions

**Purpose**: Specify which OSM features are treated as candidate urban-village references and which road features are retained as morphology context.  
**Input**: city-level PBF objects parsed by `pyrosm`.  
**Output**: helper functions that return cleaned candidate and road GeoDataFrames.


In [ ]:
# ----------------------------------------------------------------------------
# 6. Helper functions for parsing each city PBF with pyrosm
# ----------------------------------------------------------------------------

CANDIDATE_KEEP_COLS = [
    "city", "candidate_source", "id", "osm_type",
    "place", "landuse", "residential", "name", "geometry",
]
ROAD_KEEP_COLS = [
    "city", "id", "osm_type", "highway", "name", "geometry",
]


def to_clean_gdf(df, city, source, keep_cols):
    if df is None or len(df) == 0:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    out = df.copy()
    out["city"] = city
    if source is not None:
        out["candidate_source"] = source
    for col in keep_cols:
        if col not in out.columns:
            out[col] = None
    out = out[keep_cols]
    out = out[out.geometry.notna() & ~out.geometry.is_empty]
    for c in out.columns:
        if c != "geometry":
            out[c] = out[c].astype(str)
    return gpd.GeoDataFrame(out, geometry="geometry", crs="EPSG:4326")


def extract_candidates(osm, city):
    places = osm.get_data_by_custom_criteria(
        custom_filter={"place": ["village", "hamlet", "locality"]},
        filter_type="keep",
        keep_nodes=True, keep_ways=True, keep_relations=True,
    )
    places_clean = to_clean_gdf(
        places, city, "place_village_hamlet_locality", CANDIDATE_KEEP_COLS,
    )

    try:
        landuse = osm.get_landuse(extra_attributes=["residential"])
    except Exception:
        landuse = osm.get_landuse()
    rural_clean = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    if landuse is not None and len(landuse) > 0 and "residential" in landuse.columns:
        rural = landuse[landuse["residential"].astype(str).str.lower() == "rural"]
        rural_clean = to_clean_gdf(rural, city, "residential_rural", CANDIDATE_KEEP_COLS)

    pieces = [df for df in [places_clean, rural_clean] if len(df) > 0]
    if not pieces:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    cand = pd.concat(pieces, ignore_index=True)
    return gpd.GeoDataFrame(cand, geometry="geometry", crs="EPSG:4326")


def extract_roads(osm, city):
    network = osm.get_network(network_type="all")
    if network is None or len(network) == 0:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    roads = to_clean_gdf(network, city, None, ROAD_KEEP_COLS)
    roads = roads[roads.geometry.geom_type.isin(["LineString", "MultiLineString"])]
    return roads


### Step 8: Extract, clip, and save candidates and roads

**Purpose**: Parse each city PBF, clip features to the exact urban-core polygon, and export the OSM layers used by Book 2.  
**Input**: per-city PBFs and `CITY_POLYGONS`.  
**Output**: `{city}_candidates.geojson` and `{city}_roads.geojson` for all available cities.


In [ ]:
# ----------------------------------------------------------------------------
# 7. Main extraction loop  (clip uses urban_cores polygon, not GADM)
# ----------------------------------------------------------------------------

candidate_summary = []
road_summary = []

for city, pbf_path in city_pbf_paths.items():
    print(f"\n=== {city} ===")
    t0 = time.time()

    osm = pyrosm.OSM(str(pbf_path))

    candidates = extract_candidates(osm, city)
    roads = extract_roads(osm, city)

    # Clip to the urban_cores polygon (same AOI as Book 1's grid).
    poly = CITY_POLYGONS[city]
    poly_gdf = gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326")
    if len(candidates) > 0:
        candidates = gpd.clip(candidates, poly_gdf)
    if len(roads) > 0:
        roads = gpd.clip(roads, poly_gdf)

    cand_path = OSM_CANDIDATE_DIR / f"{city}_candidates.geojson"
    road_path = OSM_ROAD_DIR / f"{city}_roads.geojson"

    if len(candidates) > 0 and OVERWRITE_GEOJSON:
        candidates.to_file(cand_path, driver="GeoJSON")
    if len(roads) > 0 and OVERWRITE_GEOJSON:
        roads.to_file(road_path, driver="GeoJSON")

    elapsed = round(time.time() - t0, 1)
    candidate_summary.append({
        "city": city,
        "n_candidates": len(candidates),
        "n_place_pois": int((candidates["candidate_source"] == "place_village_hamlet_locality").sum()) if len(candidates) > 0 else 0,
        "n_rural_residential": int((candidates["candidate_source"] == "residential_rural").sum()) if len(candidates) > 0 else 0,
        "took_s": elapsed,
        "status": "downloaded" if len(candidates) > 0 else "empty",
    })
    road_summary.append({
        "city": city,
        "n_roads": len(roads),
        "took_s": elapsed,
        "status": "downloaded" if len(roads) > 0 else "empty",
    })

    print(f"  candidates: {len(candidates)}")
    print(f"  roads:      {len(roads)}")
    print(f"  took:       {elapsed}s")



=== guangzhou ===
  candidates: 409
  roads:      105639
  took:       128.9s

=== shenzhen ===
  candidates: 713
  roads:      98880
  took:       88.8s

=== tianjin ===
  candidates: 251
  roads:      65011
  took:       53.8s

=== beijing ===
  candidates: 1750
  roads:      129829
  took:       108.1s

=== xian ===
  candidates: 263
  roads:      27875
  took:       36.8s

=== dongguan ===
  candidates: 711
  roads:      42863
  took:       46.0s

=== shanghai ===
  candidates: 417
  roads:      192008
  took:       126.6s

=== chengdu ===
  candidates: 132
  roads:      57981
  took:       46.0s

=== chongqing ===
  candidates: 111
  roads:      30431
  took:       37.0s

=== wuhan ===
  candidates: 57
  roads:      29010
  took:       19.5s


### Step 9: Save extraction summary tables

**Purpose**: Record candidate and road counts so OSM coverage can be audited before modelling.  
**Input**: extraction counters collected during city processing.  
**Output**: `book0_candidate_summary.csv` and `book0_road_summary.csv`.


In [ ]:
# ----------------------------------------------------------------------------
# 8. Save summary logs
# ----------------------------------------------------------------------------

cand_df = pd.DataFrame(candidate_summary)
road_df = pd.DataFrame(road_summary)

cand_df.to_csv(LOG_DIR / "book0_candidate_summary.csv", index=False)
road_df.to_csv(LOG_DIR / "book0_road_summary.csv", index=False)

summary = cand_df.merge(road_df, on="city", suffixes=("", "_road"))

print("\n=== SUMMARY ===")
display(summary)



=== SUMMARY ===


,city,n_candidates,n_place_pois,n_rural_residential,took_s,status,n_roads,took_s_road,status_road
0,guangzhou,409,393,16,128.9,downloaded,105639,128.9,downloaded
1,shenzhen,713,674,39,88.8,downloaded,98880,88.8,downloaded
2,tianjin,251,250,1,53.8,downloaded,65011,53.8,downloaded
3,beijing,1750,1739,11,108.1,downloaded,129829,108.1,downloaded
4,xian,263,203,60,36.8,downloaded,27875,36.8,downloaded
5,dongguan,711,709,2,46.0,downloaded,42863,46.0,downloaded
6,shanghai,417,413,4,126.6,downloaded,192008,126.6,downloaded
7,chengdu,132,126,6,46.0,downloaded,57981,46.0,downloaded
8,chongqing,111,111,0,37.0,downloaded,30431,37.0,downloaded
9,wuhan,57,53,4,19.5,downloaded,29010,19.5,downloaded


### Step 10: Check for low-coverage cities

**Purpose**: Flag cities where OSM candidates or roads are too sparse to support reliable weak supervision or morphology features.  
**Input**: merged candidate and road summary tables.  
**Output**: diagnostic tables printed in the notebook for quality control.


In [ ]:
# ----------------------------------------------------------------------------
# 9. Sanity checks
# ----------------------------------------------------------------------------

print("\nLow-candidate cities (n < 10):")
low_cand = summary[summary["n_candidates"] < 10]
if len(low_cand) > 0:
    display(low_cand)
else:
    print("  none")

print("\nLow-road cities (n < 1000):")
low_road = summary[summary["n_roads"] < 1000]
if len(low_road) > 0:
    display(low_road)
else:
    print("  none")

print("\n=== Done ===")
print("Candidate dir:", OSM_CANDIDATE_DIR)
print("Road dir:     ", OSM_ROAD_DIR)
print("\nNext step: Book 1 (already done) -> Book 2 in Colab.")



Low-candidate cities (n < 10):
  none

Low-road cities (n < 1000):
  none

=== Done ===
Candidate dir: /content/drive/MyDrive/0069/week10/osm_candidates
Road dir:      /content/drive/MyDrive/0069/week10/osm_roads

Next step: Book 1 (already done) -> Book 2 in Colab.
